# 02 - Bronze Layer

**Objetivo:** Criar a camada Bronze da arquitetura Medallion.

Nesta etapa os dados brutos (raw) são carregados do arquivo Excel e convertidos para o formato Parquet, que é mais eficiente, leve e adequado para processamento analítico.

### O que é feito nesta camada:
- Leitura do arquivo original (`SerasaCredito - Case.xlsx`)
- Remoção de registros inválidos (nulos em `creditType`)
- Correção básica de tipos de dados
- Padronização dos nomes das colunas (snake_case + português)
- Tradução dos valores da coluna `tipo_credito` para português
- Salvamento em formato Parquet na pasta `data/bronze/`

**Importante:** Nesta camada **não** são feitas transformações complexas, criação de dimensões ou fato.  
O foco é apenas garantir que os dados estejam limpos, consistentes e prontos para a próxima camada (Silver).

**Próxima camada:** [03_silver_layer.ipynb](./03_silver_layer.ipynb) -> Modelagem dimensional (Star Schema)

In [1]:
import pandas as pd
from pathlib import Path

In [5]:
# ============================
# 02 - BRONZE LAYER
# ============================

# 1. Caminhos
raw_path = Path('../data/raw/SerasaCredito - Case.xlsx')
bronze_path = Path('../data/bronze/credit_data_bronze.parquet')

# 2. Carregar raw
df = pd.read_excel(raw_path)

# 3. Limpeza + renomeação para português (padrão snake_case)
df = df.dropna(subset=['creditType']).copy()

df = df.rename(columns={
    'data':                  'data_registro',
    'fx_score_de_credito':   'faixa_score_credito',
    'fx_de_renda':           'faixa_renda',
    'Negativado':            'flag_negativado',
    'faixa_etaria':          'faixa_etaria',
    'total_users_simulando': 'total_usuarios_simulando',
    'creditType':            'tipo_credito',
    'usuario_elegivel':      'usuarios_elegiveis',
    'possui_oferta':         'possui_oferta',
    'convercao_efetiva':     'conversao_efetiva',   # corrige o erro que você tinha
    'receita_gerada':        'receita_gerada'
})

# 4. Mapeamento dos tipos de crédito para português
mapeamento_credito = {
    'digital-account':        'conta_digital',
    'personal-loan-fgts':     'emprestimo_pessoal_fgts',
    'personal-loan':          'emprestimo_pessoal',
    'vehicle-guarantee-loan': 'emprestimo_garantia_veiculo',
    'credit-card':            'cartao_credito',
    'payroll-loan':           'emprestimo_consignado',
    'home-equity':            'emprestimo_garantia_imovel'
}

df['tipo_credito'] = df['tipo_credito'].map(mapeamento_credito)

# 5. Correção de data types
df['data_registro'] = pd.to_datetime(df['data_registro'])
df['flag_negativado'] = df['flag_negativado'].astype(bool)

# 6. Corrigir nome da coluna (você tinha um erro de digitação)
df.rename(columns={'convercao_efetiva': 'conversao_efetiva'}, inplace=True)

# 7. (Opcional) Padronizar strings
df['faixa_score_credito'] = df['faixa_score_credito'].str.strip()
df['faixa_renda'] = df['faixa_renda'].str.strip()
df['faixa_etaria'] = df['faixa_etaria'].str.strip()
df['tipo_credito'] = df['tipo_credito'].str.strip()

In [ ]:
# 8. Salvar em Bronze
bronze_path.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(bronze_path, index=False)

print(f"Bronze layer criada com sucesso! {len(df):,} linhas")
print(df.dtypes)

Bronze layer criada com sucesso! 105,651 linhas
data_registro               datetime64[ns]
faixa_score_credito                 object
faixa_renda                         object
flag_negativado                       bool
faixa_etaria                        object
total_usuarios_simulando             int64
tipo_credito                        object
usuarios_elegiveis                 float64
possui_oferta                      float64
conversao_efetiva                  float64
receita_gerada                     float64
dtype: object
